In [2]:
# Cell 1 — Setup
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.data_split import load_features, time_based_split
from src.models.evaluate import evaluate_predictions, evaluate_by_group
from src.models.baseline_driver_rolling import predict_driver_rolling
from src.models.baseline_circuit_history import predict_circuit_history
from src.models.baseline_team_median import predict_team_median

df = load_features()
train, val, test = time_based_split(df)

FileNotFoundError: [Errno 2] No such file or directory: 'data\\features\\qualifying_features.parquet'

In [ ]:
# Cell 2 — Run all three baselines on validation set
val = val.copy()
val["Pred_A_DriverRolling"] = predict_driver_rolling(val).fillna(val["DeltaToFastest_s"].median())
val["Pred_B_CircuitHistory"] = predict_circuit_history(val).fillna(val["DeltaToFastest_s"].median())
val["Pred_C_TeamMedian"] = predict_team_median(val).fillna(val["DeltaToFastest_s"].median())

results = []
results.append(evaluate_predictions(val["DeltaToFastest_s"], val["Pred_A_DriverRolling"], "A - Driver Rolling"))
results.append(evaluate_predictions(val["DeltaToFastest_s"], val["Pred_B_CircuitHistory"], "B - Circuit History"))
results.append(evaluate_predictions(val["DeltaToFastest_s"], val["Pred_C_TeamMedian"], "C - Team Median"))

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Cell 3 — Visual comparison
import plotly.express as px

fig = px.bar(
    results_df, x="Model", y="MAE",
    title="Baseline Model Comparison — Mean Absolute Error",
    labels={"MAE": "MAE (seconds)"},
    text="MAE"
)
fig.update_traces(textposition="outside")
fig.show()

In [ ]:
# Cell 4 — Which baseline wins on which circuits?
best_baseline = "Pred_C_TeamMedian"  # update to whichever wins overall
by_circuit = evaluate_by_group(val, "DeltaToFastest_s", best_baseline, "EventName")
by_circuit.head(10)

In [ ]:
# Cell 5 — Save the best baseline's metrics for later comparison against XGBoost
best_metrics = results_df.loc[results_df["MAE"].idxmin()]
best_metrics.to_frame().T.to_csv(PROJECT_ROOT / "models/metrics/baseline_best.csv", index=False)
print(f"Best baseline: {best_metrics['Model']} — MAE {best_metrics['MAE']}s")